In [ ]:
#!/usr/bin/env python3
"""
MultiGrate + Cox survival TEST pipeline for TCGA-BRCA.

Run train_multigrate_brca_survival.ipynb first. This notebook then:
  - loads the saved MultiVAE model
  - builds a TEST AnnData object from the frozen BRCA split
  - computes MultiGrate embeddings
  - applies the saved embedding scaler and Cox coefficients
  - saves AMP-compatible survival outputs
"""

from __future__ import annotations
from pathlib import Path
from datetime import datetime
import json

import numpy as np
import pandas as pd
import anndata as ad
import multigrate as mtg
import scvi
import joblib


In [ ]:
# -----------------------------
# Config
# -----------------------------

matrices_dir = Path("../matrices")
data_dir     = Path("../data")
splits_dir   = Path("../splits")
model_dir    = Path("../models/multigrate_survival")
results_dir  = Path("../results/brca_survival/multigrate_test")
results_dir.mkdir(parents=True, exist_ok=True)

SPLIT_TAG = "brca_survival"
SEED = 0

model_root = model_dir / f"{SPLIT_TAG}_multigrate_cox"
vae_dir = model_root / "multivae_model"
scaler_path = model_root / "embedding_scaler.pkl"
cox_coef_path = model_root / "cox_coefficients.csv"

print("Model root:", model_root.resolve())
print("Results dir:", results_dir.resolve())


In [ ]:
# -----------------------------
# Helpers
# -----------------------------

def load_index_csv_0based(path: Path, n_total: int) -> np.ndarray:
    df = pd.read_csv(path)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) == 0:
        raise ValueError(f"No numeric index column in {path}")
    idx = df[num_cols[0]].to_numpy()
    if np.isnan(idx).any():
        raise ValueError(f"NaN index in {path}")
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        raise ValueError(f"Non-integer index in {path}")
    mn, mx = int(idx_int.min()), int(idx_int.max())
    if mn == 0 and mx == n_total - 1:
        idx0 = idx_int
    elif mn == 1 and mx == n_total:
        idx0 = idx_int - 1
    else:
        idx0 = idx_int
    if (idx0 < 0).any() or (idx0 >= n_total).any():
        raise ValueError(f"Out-of-range index in {path}: [{idx0.min()}, {idx0.max()}], n={n_total}")
    return idx0


def matrix_to_anndata(X: np.ndarray, obs_names: list[str], var_names: list[str]) -> ad.AnnData:
    A = ad.AnnData(X=X.astype(np.float32))
    A.obs_names = pd.Index(obs_names).astype(str)
    A.var_names = pd.Index(var_names).astype(str)
    A.layers["norm"] = A.X.copy()
    return A


def build_multigrate_adata(rna: ad.AnnData, meth: ad.AnnData, cnv: ad.AnnData):
    meth.X = meth.layers["norm"].astype(np.float32)
    cnv.X = cnv.layers["norm"].astype(np.float32)
    adatas = [[rna], [meth], [cnv]]
    adata = mtg.data.organize_multimodal_anndatas(
        adatas=adatas,
        layers=[["norm"], ["norm"], ["norm"]],
    )
    return adata, rna.shape[1]


def concordance_index_survival(time: np.ndarray, event: np.ndarray, log_risk: np.ndarray) -> float:
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    score = -np.asarray(log_risk, dtype=float)
    concordant = 0.0
    permissible = 0.0
    n = len(time)
    for i in range(n):
        for j in range(i + 1, n):
            if time[i] == time[j]:
                continue
            if time[i] < time[j] and event[i] == 1:
                permissible += 1.0
                if score[i] < score[j]:
                    concordant += 1.0
                elif score[i] == score[j]:
                    concordant += 0.5
            elif time[j] < time[i] and event[j] == 1:
                permissible += 1.0
                if score[j] < score[i]:
                    concordant += 1.0
                elif score[j] == score[i]:
                    concordant += 0.5
    return float(concordant / permissible) if permissible else float("nan")


In [ ]:
# -----------------------------
# Load frozen split and TEST data
# -----------------------------

for p in [vae_dir, scaler_path, cox_coef_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {p}. Run train_multigrate_brca_survival.ipynb first.")

sample_ids = pd.read_csv(splits_dir / f"{SPLIT_TAG}_sample_ids.csv")["sample_id"].tolist()
idx_te = load_index_csv_0based(splits_dir / f"{SPLIT_TAG}_test_idx.csv", len(sample_ids))
test_ids = [sample_ids[i] for i in idx_te]

rna_df  = pd.read_csv(matrices_dir / "RNA_X_full.csv", index_col=0)
meth_df = pd.read_csv(matrices_dir / "Methylation_X_full.csv", index_col=0)
cnv_df  = pd.read_csv(matrices_dir / "CNV_X_ld_features.csv", index_col=0)

surv_raw = pd.read_csv(data_dir / "BRCA_survival.tsv", sep="\t", index_col=0)
surv_df = surv_raw[["OS", "OS.time"]].copy()
surv_df.columns = ["event", "time"]
surv_df["event"] = pd.to_numeric(surv_df["event"], errors="coerce")
surv_df["time"] = pd.to_numeric(surv_df["time"], errors="coerce")
surv_df = surv_df.dropna(subset=["event", "time"])
surv_df = surv_df[surv_df["time"] > 0]

rna_te = matrix_to_anndata(rna_df.loc[test_ids].to_numpy(dtype=np.float32), test_ids, list(rna_df.columns))
meth_te = matrix_to_anndata(meth_df.loc[test_ids].to_numpy(dtype=np.float32), test_ids, list(meth_df.columns))
cnv_te = matrix_to_anndata(cnv_df.loc[test_ids].to_numpy(dtype=np.float32), test_ids, list(cnv_df.columns))
event_te = surv_df.loc[test_ids, "event"].to_numpy(dtype=np.float32)
time_te = surv_df.loc[test_ids, "time"].to_numpy(dtype=np.float32)

print(f"Loaded TEST samples: {len(test_ids)} | events: {int(event_te.sum())}")


In [ ]:
# -----------------------------
# Load MultiVAE and score Cox risk
# -----------------------------

scvi.settings.seed = SEED
adata_te, rna_end = build_multigrate_adata(rna_te, meth_te, cnv_te)
mtg.model.MultiVAE.setup_anndata(adata_te, rna_indices_end=rna_end)
vae = mtg.model.MultiVAE.load(str(vae_dir), adata=adata_te)

try:
    Z_test = np.asarray(vae.get_latent_representation(), dtype=np.float32)
except Exception:
    vae.get_model_output()
    if "X_multigrate" not in adata_te.obsm:
        raise RuntimeError("Missing adata_te.obsm['X_multigrate'] after get_model_output()")
    Z_test = np.asarray(adata_te.obsm["X_multigrate"], dtype=np.float32)

scaler = joblib.load(scaler_path)
Z_test_s = scaler.transform(Z_test).astype(np.float32)

coef_df = pd.read_csv(cox_coef_path)
coef_df = coef_df.sort_values("factor", key=lambda s: s.str.extract(r"(\d+)")[0].astype(int))
beta = coef_df["coef"].to_numpy(dtype=np.float64)
if Z_test_s.shape[1] != len(beta):
    raise ValueError(f"Embedding/coef mismatch: Z_test_s has {Z_test_s.shape[1]} columns, beta has {len(beta)}")

log_risk_test = Z_test_s @ beta
c_test = concordance_index_survival(time_te, event_te, log_risk_test)
print(f"MultiGrate + Cox TEST C-index: {c_test:.4f}")


In [ ]:
# -----------------------------
# Save outputs
# -----------------------------

np.save(results_dir / "test_embedding.npy", Z_test.astype(np.float32))
np.save(results_dir / "test_embedding_scaled.npy", Z_test_s.astype(np.float32))
np.save(results_dir / "log_risk_test.npy", log_risk_test.astype(np.float32))
np.save(results_dir / "event_test.npy", event_te)
np.save(results_dir / "time_test.npy", time_te)
np.save(results_dir / "idx_test.npy", idx_te.astype(int))

pred_df = pd.DataFrame({
    "sample_id": test_ids,
    "time": time_te,
    "event": event_te,
    "log_risk": log_risk_test,
    "risk_group": np.where(log_risk_test >= np.median(log_risk_test), "High risk", "Low risk"),
})
pred_df.to_csv(results_dir / "test_predictions.csv", index=False)

report = {
    "c_index_test": float(c_test),
    "n_test": int(len(test_ids)),
    "n_events_test": int(event_te.sum()),
    "run": {
        "method": "multigrate+linear_cox",
        "task": "survival",
        "split_tag": SPLIT_TAG,
        "vae_dir": str(vae_dir),
        "scaler": str(scaler_path),
        "cox_coefficients": str(cox_coef_path),
        "views": ["RNA", "Methylation", "CNV"],
    },
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}
with open(results_dir / "metrics.json", "w") as f:
    json.dump(report, f, indent=2)

print("Saved to:", results_dir.resolve())
print(json.dumps(report, indent=2))
